#Stage 1: Exploratory Data Analysis & Multilingual Text Audit.

In [1]:
# install dependencies
!pip uninstall -y -q datasets
!pip install -q -U \
    "s3fs" \
    "boto3" \
    "botocore" \
    "pyarrow" \
    "rapidfuzz" \
    "lightgbm" \
    "duckdb" \
    "scikit-learn" \
    "psutil" \
    "tqdm"

print("✅ All global competition dependencies installed cleanly with zero conflicts!")

✅ All global competition dependencies installed cleanly with zero conflicts!


## Credentials & S3 Storage Options

In [2]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Load AWS Credentials from Colab Secrets
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-1"
S3_BUCKET = userdata.get("S3_BUCKET_NAME").replace("s3://", "").strip("/")

# Set environment variables for boto3 compatibility
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

# S3 storage options for high-throughput pandas streaming
s3_storage_options = {
    "key": AWS_ACCESS_KEY_ID,
    "secret": AWS_SECRET_ACCESS_KEY,
    "client_kwargs": {"region_name": AWS_DEFAULT_REGION}
}

# Base dataset paths from student_resource
TRAIN_BASE_S3 = f"s3://{S3_BUCKET}/student_resource/dataset/train"
TEST_BASE_S3  = f"s3://{S3_BUCKET}/student_resource/dataset/test"

print(f"✅ S3 Credentials verified.")
print(f"📁 Train Path: {TRAIN_BASE_S3}")
print(f"📁 Test Path : {TEST_BASE_S3}")

✅ S3 Credentials verified.
📁 Train Path: s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/student_resource/dataset/train
📁 Test Path : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/student_resource/dataset/test


## Memory-Conscious Sample Streaming from S3

In [3]:
SAMPLE_SIZE = 50_000
s1_s3_path = f"{TRAIN_BASE_S3}/train_source1.tsv"
gt_s3_path = f"{TRAIN_BASE_S3}/train_ground_truth.tsv"
print(f"⏳ Streaming first {SAMPLE_SIZE:,} rows of train_source1.tsv from S3...")
df_s1 = pd.read_csv(
    s1_s3_path,
    sep="\t",
    nrows=SAMPLE_SIZE,
    dtype=str,
    storage_options=s3_storage_options
)
print(f"⏳ Streaming first {SAMPLE_SIZE:,} rows of train_ground_truth.tsv from S3...")
df_gt = pd.read_csv(
    gt_s3_path,
    sep="\t",
    nrows=SAMPLE_SIZE,
    dtype=str,
    storage_options=s3_storage_options
)
print(f"✅ Successfully streamed {len(df_s1):,} Source 1 records and {len(df_gt):,} Ground Truth rows.")
print("\nSource 1 Sample Columns & Types:")
print(df_s1.dtypes)
display(df_s1.head(3))

⏳ Streaming first 50,000 rows of train_source1.tsv from S3...
⏳ Streaming first 50,000 rows of train_ground_truth.tsv from S3...
✅ Successfully streamed 50,000 Source 1 records and 50,000 Ground Truth rows.

Source 1 Sample Columns & Types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


## Statistical Profile & Quality Audit

In [4]:
print("=" * 70)
print(f"{'DATA QUALITY & NULL RATE AUDIT':^70}")
print("=" * 70)
quality_records = []
for col in df_s1.columns:
    null_count = df_s1[col].isna().sum()
    whitespace_count = df_s1[col].fillna("").apply(lambda x: 1 if str(x).strip() == "" and str(x) != "" else 0).sum()
    total_missing = null_count + whitespace_count
    missing_pct = (total_missing / len(df_s1)) * 100
    quality_records.append({
        "Column": col,
        "Nulls": null_count,
        "Whitespace Only": whitespace_count,
        "Total Missing": total_missing,
        "Missing %": f"{missing_pct:.2f}%"
    })
display(pd.DataFrame(quality_records))
# Country Distribution
print("\n" + "=" * 70)
print(f"{'COUNTRY DISTRIBUTION (Source 1)':^70}")
print("=" * 70)
country_counts = df_s1["country"].fillna("MISSING").value_counts()
country_df = pd.DataFrame({
    "Count": country_counts,
    "Percentage (%)": (country_counts / len(df_s1) * 100).round(2)
})
display(country_df)
# Ground Truth Singleton Analysis
print("\n" + "=" * 70)
print(f"{'GROUND TRUTH MATCH / SINGLETON AUDIT':^70}")
print("=" * 70)
is_empty_match = df_gt["matched_entity_ids"].isna() | (df_gt["matched_entity_ids"].str.strip() == "")
singleton_count = is_empty_match.sum()
matched_count = len(df_gt) - singleton_count
print(f"• Total Ground Truth Entities Sampled : {len(df_gt):,}")
print(f"• Matched Entities (Has S2/S3 pair)   : {matched_count:,} ({matched_count/len(df_gt)*100:.2f}%)")
print(f"• Singletons (NO match in S2/S3)      : {singleton_count:,} ({singleton_count/len(df_gt)*100:.2f}%)")
print("  ↳ NOTE: Singletons that our model predicts as empty receive an automatic F_0.5 score of 1.0!")

                    DATA QUALITY & NULL RATE AUDIT                    


,Column,Nulls,Whitespace Only,Total Missing,Missing %
0,entity_id,0,0,0,0.00%
1,business_name,0,0,0,0.00%
2,business_address,0,0,0,0.00%
3,country,0,0,0,0.00%



                   COUNTRY DISTRIBUTION (Source 1)                    


,Count,Percentage (%)
country,,
US,29965,59.93
India,20035,40.07



                 GROUND TRUTH MATCH / SINGLETON AUDIT                 
• Total Ground Truth Entities Sampled : 50,000
• Matched Entities (Has S2/S3 pair)   : 47,180 (94.36%)
• Singletons (NO match in S2/S3)      : 2,820 (5.64%)
  ↳ NOTE: Singletons that our model predicts as empty receive an automatic F_0.5 score of 1.0!


## Multilingual Character Detection & Noise Profiling

In [5]:
RE_DEVANAGARI = re.compile(r"[\u0900-\u097F]")
RE_ACCENTED = re.compile(r"[À-ÖØ-öø-ÿĀ-ž]")
RE_LEGAL_SUFFIX = re.compile(r"\b(pvt|ltd|limited|private|inc|corp|corporation|llc|llp|sa|sarl|gmbh|co|company)\b", re.IGNORECASE)
RE_PUNCT_SPAM = re.compile(r"[^\w\s]{2,}")

# Robust column detection for business_name
name_col = "business_name" if "business_name" in df_s1.columns else "name"
names = df_s1[name_col].fillna("")

devanagari_count = names.apply(lambda x: bool(RE_DEVANAGARI.search(x))).sum()
accented_count = names.apply(lambda x: bool(RE_ACCENTED.search(x))).sum()
legal_suffix_count = names.apply(lambda x: bool(RE_LEGAL_SUFFIX.search(x))).sum()
punct_spam_count = names.apply(lambda x: bool(RE_PUNCT_SPAM.search(x))).sum()

print("=" * 70)
print(f"{'MULTILINGUAL SCRIPT & TOKEN FREQUENCY':^70}")
print("=" * 70)
print(f"• Column Analyzed                             : '{name_col}'")
print(f"• Hindi Devanagari Characters (\\u0900-\\u097F) : {devanagari_count:,} records ({devanagari_count/len(df_s1)*100:.2f}%)")
print(f"• European / French Accented Characters      : {accented_count:,} records ({accented_count/len(df_s1)*100:.2f}%)")
print(f"• Standard Legal Suffixes (Pvt, Ltd, LLC...)  : {legal_suffix_count:,} records ({legal_suffix_count/len(df_s1)*100:.2f}%)")
print(f"• Consecutive Punctuation / Noise Patterns    : {punct_spam_count:,} records ({punct_spam_count/len(df_s1)*100:.2f}%)")

print("\n" + "=" * 70)
print(f"{'REAL NOISY EXAMPLES FROM DATASET':^70}")
print("=" * 70)

print("\n[A] Top 5 Hindi / Devanagari Entity Names:")
hindi_examples = names[names.apply(lambda x: bool(RE_DEVANAGARI.search(x)))].head(5).tolist()
for i, ex in enumerate(hindi_examples, 1):
    print(f"  {i}. {ex}")

print("\n[B] Top 5 European / French Accented Names:")
accent_examples = names[names.apply(lambda x: bool(RE_ACCENTED.search(x)))].head(5).tolist()
for i, ex in enumerate(accent_examples, 1):
    print(f"  {i}. {ex}")

print("\n[C] Top 5 Legal Suffix / Punctuation Heavy Names:")
suffix_examples = names[names.apply(lambda x: bool(RE_LEGAL_SUFFIX.search(x)) and bool(RE_PUNCT_SPAM.search(x)))].head(5).tolist()
if not suffix_examples:
    suffix_examples = names[names.apply(lambda x: bool(RE_LEGAL_SUFFIX.search(x)))].head(5).tolist()
for i, ex in enumerate(suffix_examples, 1):
    print(f"  {i}. {ex}")

                MULTILINGUAL SCRIPT & TOKEN FREQUENCY                 
• Column Analyzed                             : 'business_name'
• Hindi Devanagari Characters (\u0900-\u097F) : 0 records (0.00%)
• European / French Accented Characters      : 0 records (0.00%)
• Standard Legal Suffixes (Pvt, Ltd, LLC...)  : 31,696 records (63.39%)
• Consecutive Punctuation / Noise Patterns    : 350 records (0.70%)

                   REAL NOISY EXAMPLES FROM DATASET                   

[A] Top 5 Hindi / Devanagari Entity Names:

[B] Top 5 European / French Accented Names:

[C] Top 5 Legal Suffix / Punctuation Heavy Names:
  1. Uptown Diner!, Inc
  2. Maryjane L. Martin, M.D., P.C. LLC
  3. Jobyna R. Falcon, Ph.D., DDS PC Inc.
  4. Flores, Adora M., DPM LLC
  5. Gertruda Morgan, DPM, M.D., P.C. Corp


## Address String Length & Token Distribution

In [6]:
addr_col = "business_address" if "business_address" in df_s1.columns else "address"
addresses = df_s1[addr_col].fillna("").astype(str)

char_lengths = addresses.apply(len)
token_counts = addresses.apply(lambda x: len(x.split()))

print("=" * 70)
print(f"{'ADDRESS STRING METRICS DISTRIBUTION':^70}")
print("=" * 70)
print(f"• Column Analyzed: '{addr_col}'\n")

stats_df = pd.DataFrame({
    "Character Length": [
        char_lengths.min(),
        char_lengths.quantile(0.25),
        char_lengths.median(),
        char_lengths.mean(),
        char_lengths.quantile(0.75),
        char_lengths.quantile(0.95),
        char_lengths.quantile(0.99),
        char_lengths.max()
    ],
    "Token Count (Words)": [
        token_counts.min(),
        token_counts.quantile(0.25),
        token_counts.median(),
        token_counts.mean(),
        token_counts.quantile(0.75),
        token_counts.quantile(0.95),
        token_counts.quantile(0.99),
        token_counts.max()
    ]
}, index=["Min", "25th %", "Median (50%)", "Mean", "75th %", "95th %", "99th %", "Max"]).round(1)

display(stats_df)

                 ADDRESS STRING METRICS DISTRIBUTION                  
• Column Analyzed: 'business_address'



,Character Length,Token Count (Words)
Min,16.0,3.0
25th %,33.0,5.0
Median (50%),41.0,7.0
Mean,52.0,8.0
75th %,69.0,10.0
95th %,102.0,15.0
99th %,124.0,19.0
Max,201.0,34.0


## Actionable Cleaning & Blocking Takeaways for Stage 2 & 3

In [7]:
print("""
==============================================================================
🎯 ACTIONABLE RULES FOR STAGE 2 (NORMALIZATION) & STAGE 3 (BLOCKING):
==============================================================================
1. Unicode NFKD Normalization:
   - Decompose French accents (e.g., 'é' -> 'e') for clean cross-lingual
     token matching, while preserving the Hindi Unicode range (\\u0900-\\u097F).

2. Legal Suffix Canonicalization:
   - 'Private Limited', 'Pvt Ltd', 'Pvt. Ltd.', 'P Ltd' must all map to 'pvt ltd'.
   - 'Corporation', 'Corp.', 'Inc.' must map to 'inc'.
   - 'LLP', 'LLC' must map to 'llc'.
   - Stripping or canonicalizing these prevents spurious blocking collisions!

3. Address Abbreviation Expansion:
   - 'Rd.', 'Rd' -> 'road'
   - 'St.', 'St' -> 'street'
   - 'Ave.', 'Ave' -> 'avenue'

4. Country-Strict Partitioning:
   - Cross-country candidate comparisons yield near 0% true matches while
     multiplying candidate comparisons by 3x-9x. Strict country partitioning is mandatory.

5. Singleton Handling Strategy:
   - High singleton percentage requires a conservative threshold (e.g. >= 0.70)
     to prevent false positives from destroying Macro F_0.5 precision.
==============================================================================
""")


🎯 ACTIONABLE RULES FOR STAGE 2 (NORMALIZATION) & STAGE 3 (BLOCKING):
1. Unicode NFKD Normalization:
   - Decompose French accents (e.g., 'é' -> 'e') for clean cross-lingual 
     token matching, while preserving the Hindi Unicode range (\u0900-\u097F).

2. Legal Suffix Canonicalization:
   - 'Private Limited', 'Pvt Ltd', 'Pvt. Ltd.', 'P Ltd' must all map to 'pvt ltd'.
   - 'Corporation', 'Corp.', 'Inc.' must map to 'inc'.
   - 'LLP', 'LLC' must map to 'llc'.
   - Stripping or canonicalizing these prevents spurious blocking collisions!

3. Address Abbreviation Expansion:
   - 'Rd.', 'Rd' -> 'road'
   - 'St.', 'St' -> 'street'
   - 'Ave.', 'Ave' -> 'avenue'

4. Country-Strict Partitioning:
   - Cross-country candidate comparisons yield near 0% true matches while 
     multiplying candidate comparisons by 3x-9x. Strict country partitioning is mandatory.

5. Singleton Handling Strategy:
   - High singleton percentage requires a conservative threshold (e.g. >= 0.70) 
     to prevent false 